# Publication Figures

Generate clean, publication-ready figures from all experimental results.
Assumes notebooks 01-05 have been run and results are saved in `outputs/`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import pandas as pd

from persona_steering.config import VECTORS_DIR, FIGURES_DIR, Trait
from persona_steering.analysis import (
    build_transfer_matrix,
    build_per_trait_transfer,
    decompose_shared_specific,
    compute_curvature,
)
from persona_steering.utils import load_pickle, ensure_output_dirs

ensure_output_dirs()

# Publication style
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

In [ ]:
# Load data
all_vectors = load_pickle(VECTORS_DIR / "all_vectors.pkl")
persona_slugs = list(all_vectors.keys())
traits = list(Trait)

sample_trait = traits[0]
sample_layers = sorted(all_vectors[persona_slugs[0]][sample_trait].keys())
mid_layer = sample_layers[len(sample_layers) // 2]

print(f"Personas: {persona_slugs}")
print(f"Layer: {mid_layer}")

In [ ]:
# Figure 1: Transfer matrix heatmap
matrix = build_transfer_matrix(all_vectors, persona_slugs, traits, mid_layer)

fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(matrix, annot=True, fmt=".2f", xticklabels=persona_slugs,
            yticklabels=persona_slugs, cmap="viridis", vmin=0, vmax=1, ax=ax,
            square=True, linewidths=0.5)
ax.set_title("Cross-Persona Steering Vector Similarity")
plt.savefig(FIGURES_DIR / "fig1_transfer_matrix.pdf")
plt.savefig(FIGURES_DIR / "fig1_transfer_matrix.png")
plt.show()

In [ ]:
# Figure 2: Per-trait similarity breakdown
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, trait in zip(axes.flat, traits):
    m = build_per_trait_transfer(all_vectors, persona_slugs, trait, mid_layer)
    sns.heatmap(m, annot=True, fmt=".2f", xticklabels=persona_slugs,
                yticklabels=persona_slugs, cmap="viridis", vmin=0, vmax=1,
                ax=ax, square=True, linewidths=0.5)
    ax.set_title(trait.value.title())

plt.suptitle("Per-Trait Cross-Persona Similarity", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig2_per_trait_transfer.pdf")
plt.savefig(FIGURES_DIR / "fig2_per_trait_transfer.png")
plt.show()

In [ ]:
# Figure 3: Shared vs persona-specific decomposition
shared_fracs = []
for trait in traits:
    vecs = {slug: all_vectors[slug][trait][mid_layer] for slug in persona_slugs
            if trait in all_vectors[slug] and mid_layer in all_vectors[slug][trait]}
    if len(vecs) >= 2:
        decomp = decompose_shared_specific(vecs)
        shared_fracs.append({
            "trait": trait.value,
            "variance_explained": decomp.variance_explained,
        })

df_shared = pd.DataFrame(shared_fracs)

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(df_shared["trait"], df_shared["variance_explained"], color="steelblue")
ax.set_ylabel("Shared Variance Fraction")
ax.set_title("Shared vs Persona-Specific Steering Components")
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig3_shared_specific.pdf")
plt.savefig(FIGURES_DIR / "fig3_shared_specific.png")
plt.show()

In [ ]:
# Figure 4: Layer-wise similarity curves
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(traits)))

for trait, color in zip(traits, colors):
    sims_by_layer = []
    for layer in sample_layers:
        layer_sims = []
        for i, pa in enumerate(persona_slugs):
            for j, pb in enumerate(persona_slugs):
                if j <= i:
                    continue
                va = all_vectors.get(pa, {}).get(trait, {}).get(layer)
                vb = all_vectors.get(pb, {}).get(trait, {}).get(layer)
                if va is not None and vb is not None:
                    from persona_steering.utils import cosine_similarity
                    layer_sims.append(cosine_similarity(va.vector, vb.vector))
        sims_by_layer.append(np.mean(layer_sims) if layer_sims else 0)

    ax.plot(sample_layers, sims_by_layer, marker="o", markersize=3,
            label=trait.value, color=color)

ax.set_xlabel("Layer")
ax.set_ylabel("Mean Cross-Persona Cosine Similarity")
ax.set_title("Layer-wise Cross-Persona Steering Similarity")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig4_layerwise_similarity.pdf")
plt.savefig(FIGURES_DIR / "fig4_layerwise_similarity.png")
plt.show()